# V009 TCN: shifted-linear execution backend

Definitions only. Execute the unchanged V005 sequence definitions and `research/tcn_components_v009.ipynb` first. No assignment-data reads, fits, external actions or training occur on definition. The original TCN notebook is preserved.

The strict-deterministic Torch2.2.1/GTX1080Ti synthetic profile rejected the original Conv1d execution path before approved-data fitting: 3.3593 seconds/step versus .03979 baseline (84.42x), with about405MB allocated. This notebook changes only how the same kernel3 convolutions are evaluated. It does not change channels, blocks, dilation, receptive field, weights, bias, padding, dropout, head, data, objective or training budget. No validation MAE motivated the execution change. The actual-GPU <=2x baseline time and safe11GB memory gates still apply; CPU numerical parity does not establish performance.

For each output position, concatenate the zero-padded input vectors at `t-d`, `t`, and `t+d`, then apply `torch.nn.functional.linear` with `weight.permute(0,2,1).reshape(out_channels,3*in_channels)` and the original bias. This is the original symmetric Conv1d cross-correlation, evaluated by GEMM. There is no convolution, SDPA, cumulative reduction, nondeterministic fallback, altered CUDA flag, or relaxed deterministic setting in the new forward path.

`V9FastTCNPressureModel` subclasses the original model and replaces each private Conv1d module with a compatible module retaining the **same Parameter objects**. Conversion allocates no random parameter values and consumes no RNG state. Every original `state_dict` key, weight shape and bias is retained. Original and fast checkpoints load strictly into either backend. Identical starting seeds preserve all initial tensors and the RNG state before dropout. Floating-point summation differs between execution paths, so trajectories need not be bitwise equal across backends; exact resumption is required within the selected frozen backend.

APIs: `v9_fast_tcn_model(spec,n_features,prep=None)`, `v9_fast_tcn_selfcheck(device='cpu')`, and `v9_fast_tcn_source_guard(repository_root='.', expected_notebook_sha256=None)`. The original model's feature guards, exact80 length, input-only scalar contract, width96/six residual blocks, dilations1..32, LayerNorm/SiLU/dropout.15, head_width and receptive253 are inherited. This is an execution-backend repair of the same architecture package; no common recurrent-baseline initialization is claimed.

The explicit synthetic selfcheck compares standalone convolutions at all six dilations and complete TCNs against the unchanged Conv1d model on **CPU float64 and float32**, including gradients of every parameter and the inputs. Float64 tolerances are output rtol1e-9/atol1e-10 and gradient rtol1e-8/atol2e-10; float32 tolerances are output rtol2e-5/atol2e-5 and gradient rtol5e-4/atol5e-5. It also checks boundary orientation, identical parameter/state keys/counts, conversion RNG and parameter identity, strict checkpoint interchange, identical dropout RNG consumption, selected-device finite forward/backward, exact same-RNG repeatability, independent breaths, and preserved strict determinism. CUDA execution checks use the fast backend; the documented original-vs-fast oracle stays on CPU. The caller must separately run actual-GPU full-batch profiling and shared-trainer exact resume before any real-data fitting.


In [ ]:
# Required immutable model definitions and source identity. Definitions only.
_V9_FAST_TCN_REQUIRED = ('V9TCNPressureModel', 'V9_TCN_VERSION', 'V9_TCN_CHANNELS',
    'V9_TCN_DILATIONS', 'V9_TCN_RECEPTIVE_FIELD', 'canonical_model_spec',
    'seed_sequence', '_sequence_device', '_rng_state', '_restore_rng',
    'torch', 'nn', 'np', 'copy', 'hashlib', 'json', 'Path', 'V5_STEPS')
_missing_v9_fast_tcn = [name for name in _V9_FAST_TCN_REQUIRED if name not in globals()]
if _missing_v9_fast_tcn:
    raise RuntimeError('Execute unchanged V005 and original V009 TCN definitions first: ' +
        ', '.join(_missing_v9_fast_tcn))
V9_FAST_TCN_VERSION = 'v009-shifted-linear-convolution-v1'
V9_FAST_TCN_NOTEBOOK = 'research/tcn_fast_components_v009.ipynb'
V9_FAST_TCN_ORIGINAL_NOTEBOOK = 'research/tcn_components_v009.ipynb'
V9_FAST_TCN_ORIGINAL_SHA256 = 'b4a3f923cda6b3b89478e2e1ef55ddc4fccc491870eb088c1f689266e691ab9a'
V9_FAST_TCN_BASE_NOTEBOOK = 'research/sequence_components_v005.ipynb'
V9_FAST_TCN_BASE_SHA256 = 'd20e92eec5889b094a87486f876c3e1a8ae1d7eb2aa096fa5dd530b64c914b4e'
V9_FAST_TCN_CODE_SHA256 = '7a58664c22aba1a143a0dbefb653fcae9986ecb8b01c5b250058d4da4e95d64f'
V9_FAST_TCN_TOLERANCES = dict(
    float64=dict(output_rtol=1e-9, output_atol=1e-10, gradient_rtol=1e-8, gradient_atol=2e-10),
    float32=dict(output_rtol=2e-5, output_atol=2e-5, gradient_rtol=5e-4, gradient_atol=5e-5))


def v9_fast_tcn_source_guard(repository_root='.', expected_notebook_sha256=None):
    """Explicit source checks; no reads occur merely by defining this helper."""
    root = Path(repository_root)
    source = (root / V9_FAST_TCN_NOTEBOOK).read_bytes()
    book = json.loads(source)
    expected = dict(version=V9_FAST_TCN_VERSION, notebook=V9_FAST_TCN_NOTEBOOK,
        implementation_sha256=V9_FAST_TCN_CODE_SHA256)
    if book.get('metadata', {}).get('fast_tcn_component') != expected:
        raise ValueError('Fast TCN identity or implementation hash metadata changed')
    cells = [cell for cell in book['cells'] if cell.get('metadata', {}).get('v9_fast_tcn_implementation')]
    if len(cells) != 2:
        raise ValueError('Fast TCN implementation-cell layout changed')
    code_sha256 = hashlib.sha256('\n\n'.join(''.join(cell['source']) for cell in cells).encode()).hexdigest()
    if code_sha256 != V9_FAST_TCN_CODE_SHA256:
        raise ValueError('Fast TCN implementation SHA256 mismatch')
    for name, expected_hash in ((V9_FAST_TCN_ORIGINAL_NOTEBOOK, V9_FAST_TCN_ORIGINAL_SHA256),
                               (V9_FAST_TCN_BASE_NOTEBOOK, V9_FAST_TCN_BASE_SHA256)):
        if hashlib.sha256((root / name).read_bytes()).hexdigest() != expected_hash:
            raise ValueError('Fast TCN dependency SHA256 changed: ' + name)
    full_sha256 = hashlib.sha256(source).hexdigest()
    if expected_notebook_sha256 is not None and full_sha256 != expected_notebook_sha256:
        raise ValueError('Fast TCN full-notebook SHA256 mismatch')
    return dict(version=V9_FAST_TCN_VERSION, notebook=V9_FAST_TCN_NOTEBOOK,
        notebook_sha256=full_sha256, implementation_sha256=code_sha256,
        original_sha256=V9_FAST_TCN_ORIGINAL_SHA256,
        external_notebook_hash_checked=expected_notebook_sha256 is not None)


In [ ]:
# Identical kernel3 cross-correlation through shifted input vectors and GEMM.
class _V9ShiftedLinearConv1d(nn.Module):
    """Compatible B,C,T module reusing the exact original convolution parameters.

    Conversion performs no random draws, parameter initialization, or data copy.
    The only supported configuration is the original dense symmetric kernel3 path.
    """
    def __init__(self, convolution):
        super().__init__()
        if not isinstance(convolution, nn.Conv1d):
            raise TypeError('Conversion requires an original torch.nn.Conv1d')
        if (convolution.kernel_size != (3,) or convolution.stride != (1,)
                or convolution.groups != 1 or convolution.padding != convolution.dilation
                or convolution.padding_mode != 'zeros'):
            raise ValueError('Fast backend requires dense symmetric zero-padded kernel3 stride1')
        self.in_channels = convolution.in_channels
        self.out_channels = convolution.out_channels
        self.kernel_size = convolution.kernel_size
        self.stride = convolution.stride
        self.padding = convolution.padding
        self.dilation = convolution.dilation
        self.groups = convolution.groups
        self.padding_mode = convolution.padding_mode
        # Assigning an existing Parameter registers it under the identical key.
        self.weight = convolution.weight
        self.bias = convolution.bias
        self.train(convolution.training)

    def forward(self, x):
        if x.ndim != 3 or x.shape[1] != self.in_channels or x.shape[2] < 1:
            raise ValueError('Expected nonempty convolution input [B,C,T]')
        sequence = x.transpose(1, 2)
        batch, steps, channels = sequence.shape
        dilation = self.dilation[0]
        zeros = sequence.new_zeros((batch, dilation, channels))
        left = torch.cat((zeros, sequence), dim=1)[:, :steps]
        right = torch.cat((sequence, zeros), dim=1)[:, dilation:dilation + steps]
        shifted = torch.cat((left, sequence, right), dim=-1)
        # Conv1d cross-correlates [left,center,right]; it does not flip the kernel.
        flattened_weight = self.weight.permute(0, 2, 1).reshape(self.out_channels, 3 * channels)
        result = torch.nn.functional.linear(shifted, flattened_weight, self.bias)
        return result.transpose(1, 2)


class V9FastTCNPressureModel(V9TCNPressureModel):
    """Same model, parameter objects and state keys; only convolution forward differs."""
    def __init__(self, model_spec, n_features, prep=None):
        super().__init__(model_spec, n_features, prep)
        for block in self.blocks:
            block.conv1 = _V9ShiftedLinearConv1d(block.conv1)
            block.conv2 = _V9ShiftedLinearConv1d(block.conv2)


def v9_fast_tcn_model(model_spec, n_features, prep=None):
    """Shared-trainer factory for the frozen shifted-linear execution backend."""
    return V9FastTCNPressureModel(model_spec, n_features, prep)


In [ ]:
# Synthetic mathematical and execution proof only; never called on definition.
def v9_fast_tcn_selfcheck(device='cpu'):
    """CPU Conv1d oracle in two precisions plus strict selected-device fast checks.

    No assignment files/labels or real-data fits are involved. Numerical backend
    equivalence does not replace actual-GPU cost/memory and exact-resume gates.
    """
    import inspect
    import platform

    selected = _sequence_device(device)
    generator = torch.Generator().manual_seed(9090)
    rng_before = _rng_state(generator)
    backend_before = dict(threads=torch.get_num_threads(),
        deterministic=torch.are_deterministic_algorithms_enabled(),
        warn_only=torch.is_deterministic_algorithms_warn_only_enabled(),
        cudnn_benchmark=torch.backends.cudnn.benchmark,
        cudnn_deterministic=torch.backends.cudnn.deterministic,
        matmul_tf32=torch.backends.cuda.matmul.allow_tf32,
        cudnn_tf32=torch.backends.cudnn.allow_tf32)
    checks, errors = {}, {}
    def check(name, condition):
        if not bool(condition):
            raise AssertionError(name)
        checks[name] = True
    def same(left, right):
        if torch.is_tensor(left):
            return torch.is_tensor(right) and torch.equal(left, right)
        if isinstance(left, np.ndarray):
            return isinstance(right, np.ndarray) and np.array_equal(left, right)
        if isinstance(left, dict):
            return isinstance(right, dict) and left.keys() == right.keys() and all(same(left[k], right[k]) for k in left)
        if isinstance(left, (list, tuple)):
            return type(left) is type(right) and len(left) == len(right) and all(same(a, b) for a, b in zip(left, right))
        return left == right
    def rejects(name, action):
        try:
            action()
        except (ValueError, TypeError, RuntimeError):
            checks[name] = True
        else:
            raise AssertionError(name + ' did not reject')
    def compare(name, original, fast, x, tolerance):
        original.eval(); fast.eval()
        a = x.detach().clone().requires_grad_(True)
        b = x.detach().clone().requires_grad_(True)
        pa, pb = original(a), fast(b)
        check(name + '_forward', torch.allclose(pa, pb,
            rtol=tolerance['output_rtol'], atol=tolerance['output_atol']))
        probe = torch.linspace(-.7, .9, pa.numel(), dtype=pa.dtype).reshape(pa.shape)
        la = (pa*probe).mean() + .05*pa.square().mean()
        lb = (pb*probe).mean() + .05*pb.square().mean()
        original_parameters = dict(original.named_parameters())
        fast_parameters = dict(fast.named_parameters())
        check(name + '_parameter_names', original_parameters.keys() == fast_parameters.keys())
        ga = torch.autograd.grad(la, (a, *original_parameters.values()))
        gb = torch.autograd.grad(lb, (b, *fast_parameters.values()))
        check(name + '_input_gradient', torch.allclose(ga[0], gb[0],
            rtol=tolerance['gradient_rtol'], atol=tolerance['gradient_atol']))
        check(name + '_every_parameter_gradient', all(torch.allclose(left, right,
            rtol=tolerance['gradient_rtol'], atol=tolerance['gradient_atol'])
            for left, right in zip(ga[1:], gb[1:])))
        check(name + '_all_gradients_finite', all(torch.isfinite(value).all() for value in (*ga, *gb)))
        errors[name] = dict(output_max_absolute=float((pa-pb).abs().max().detach()),
            input_gradient_max_absolute=float((ga[0]-gb[0]).abs().max().detach()),
            parameter_gradient_max_absolute=max(float((left-right).abs().max().detach())
                for left, right in zip(ga[1:], gb[1:])))

    try:
        seed_sequence(20261102)
        spec = canonical_model_spec(dict(family='lstm', hidden=128, layers=2,
            dropout=.15, bidirectional=True, head_width=32))
        names = ['R', 'C', 'time_step', 'u_in', 'u_out', 'step', 'dt'] + [
            f'known_synthetic_feature_{i}' for i in range(7, 47)]
        prep = dict(feature_names=names)
        original = V9TCNPressureModel(spec, 47, prep)
        original_rng = _rng_state(generator)
        seed_sequence(20261102)
        fast = v9_fast_tcn_model(spec, 47, prep)
        fast_rng = _rng_state(generator)
        check('construction_rng_state_identical_to_original', same(original_rng, fast_rng))
        check('identical_initial_state_keys_shapes_values', original.state_dict().keys() == fast.state_dict().keys()
            and all(torch.equal(value, fast.state_dict()[name]) for name, value in original.state_dict().items()))
        parameter_count = sum(parameter.numel() for parameter in fast.parameters())
        check('production_parameter_count_unchanged_342977', parameter_count == 342977
            and parameter_count == sum(parameter.numel() for parameter in original.parameters()))
        check('all_twelve_convolutions_use_shifted_linear_backend', sum(isinstance(module,
            _V9ShiftedLinearConv1d) for module in fast.modules()) == 12
            and not any(isinstance(module, nn.Conv1d) for module in fast.modules()))
        check('architecture_dropout_and_receptive_field_unchanged', len(fast.blocks) == 6
            and fast.receptive_field == 253 and fast.head[0].in_features == 96
            and fast.head[0].out_features == 32
            and [block.dilation for block in fast.blocks] == [1, 2, 4, 8, 16, 32]
            and [module.p for module in fast.modules() if isinstance(module, nn.Dropout)] == [.15]*12)
        convolution = nn.Conv1d(5, 7, 3, padding=4, dilation=4)
        conversion_rng = _rng_state(generator)
        converted = _V9ShiftedLinearConv1d(convolution)
        check('conversion_consumes_no_rng', same(conversion_rng, _rng_state(generator)))
        check('conversion_reuses_exact_weight_and_bias_parameters', converted.weight is convolution.weight
            and converted.bias is convolution.bias)
        check('conversion_preserves_convolution_state_keys', converted.state_dict().keys() == convolution.state_dict().keys())

        # Standalone CPU oracles cover every approved dilation, T<d and bias=None.
        for dtype in (torch.float64, torch.float32):
            key = str(dtype).split('.')[-1]
            tolerance = V9_FAST_TCN_TOLERANCES[key]
            for dilation in V9_TCN_DILATIONS:
                convolution = nn.Conv1d(5, 7, 3, padding=dilation, dilation=dilation).to(dtype=dtype)
                converted = _V9ShiftedLinearConv1d(copy.deepcopy(convolution))
                x = torch.randn(2, 5, 80, dtype=dtype)
                compare(f'conv_{key}_d{dilation}', convolution, converted, x, tolerance)
            short = nn.Conv1d(2, 3, 3, padding=32, dilation=32, bias=False).to(dtype=dtype)
            compare('short_no_bias_' + key, short, _V9ShiftedLinearConv1d(copy.deepcopy(short)),
                torch.randn(2, 2, 1, dtype=dtype), tolerance)
            reference = copy.deepcopy(original).to(dtype=dtype)
            candidate = copy.deepcopy(fast).to(dtype=dtype)
            full_input = torch.randn(2, 80, 48, dtype=dtype)
            full_input[..., -1] = torch.tensor([0., 8.], dtype=dtype)[:, None]
            compare('full_model_' + key, reference, candidate, full_input, tolerance)
            candidate.load_state_dict(reference.state_dict(), strict=True)
            reference.load_state_dict(candidate.state_dict(), strict=True)
            check('strict_checkpoint_interchange_' + key,
                all(torch.equal(value, candidate.state_dict()[name]) for name, value in reference.state_dict().items()))

        # Hand-set asymmetric kernel catches an incorrect weight flattening/flip.
        hand = nn.Conv1d(1, 1, 3, padding=2, dilation=2).double()
        with torch.no_grad():
            hand.weight.copy_(torch.tensor([[[2., 3., 5.]]], dtype=torch.float64)); hand.bias.fill_(.7)
        hand_fast = _V9ShiftedLinearConv1d(copy.deepcopy(hand))
        signal = torch.arange(1., 6., dtype=torch.float64).reshape(1, 1, 5)
        expected = torch.tensor([[[18.7, 26.7, 36.7, 16.7, 21.7]]], dtype=torch.float64)
        check('asymmetric_kernel_zero_boundary_orientation', torch.allclose(hand_fast(signal), expected,
            rtol=0., atol=1e-12) and torch.allclose(hand(signal), expected, rtol=0., atol=1e-12))

        original.train(); fast.train()
        dropout_input = torch.randn(2, 80, 48); dropout_input[..., -1] = 0.
        dropout_rng = _rng_state(generator)
        with torch.no_grad():
            original_training = original(dropout_input)
            original_after = _rng_state(generator)
            _restore_rng(dropout_rng, generator)
            fast_training = fast(dropout_input)
            fast_after = _rng_state(generator)
        check('training_dropout_consumes_identical_rng', same(original_after, fast_after))
        check('same_dropout_draw_outputs_numerically_equivalent', torch.allclose(original_training, fast_training,
            rtol=V9_FAST_TCN_TOLERANCES['float32']['output_rtol'],
            atol=V9_FAST_TCN_TOLERANCES['float32']['output_atol']))

        # Only the repaired forward path executes on the requested GPU. The full
        # original Conv1d numerical oracle above is explicitly CPU, in two dtypes.
        selected_model = v9_fast_tcn_model(spec, 47, prep).to(selected)
        x = torch.randn(3, 80, 48, device=selected); x[..., -1] = 0.
        selected_model.eval()
        with torch.no_grad():
            p = selected_model(x)
            check('selected_device_finite_scalar80_output', p.shape == (3, 80) and torch.isfinite(p).all())
            check('selected_device_eval_repeat_exact', torch.equal(p, selected_model(x)))
            altered = x.clone(); altered[1, :, :47] += 2.
            other = selected_model(altered)
            check('selected_device_independent_breaths', torch.equal(p[0], other[0]) and torch.equal(p[2], other[2]))
            code_altered = x.clone(); code_altered[..., -1] = 999.
            check('internal_code_still_ignored', torch.equal(p, selected_model(code_altered)))
            selected_model(altered)
            check('selected_device_no_persistent_state', torch.equal(p, selected_model(x)))
        selected_model.train()
        backward_rng = _rng_state(generator)
        target = .2*x[..., 2] + .3
        selected_model.zero_grad(set_to_none=True)
        pa = selected_model(x); la = (pa-target).abs().mean(); la.backward()
        gradients = {name: parameter.grad.detach().clone() for name, parameter in selected_model.named_parameters()}
        check('selected_device_all_parameter_gradients_finite', torch.isfinite(la)
            and all(torch.isfinite(value).all() for value in gradients.values()))
        check('selected_device_every_shifted_kernel_receives_gradient', all(
            parameter.grad.abs().sum() > 0 for name, parameter in selected_model.named_parameters()
            if '.conv' in name and name.endswith('weight')))
        _restore_rng(backward_rng, generator)
        selected_model.zero_grad(set_to_none=True)
        pb = selected_model(x); lb = (pb-target).abs().mean(); lb.backward()
        check('selected_device_same_rng_forward_backward_exact', torch.equal(pa, pb)
            and all(torch.equal(gradients[name], parameter.grad) for name, parameter in selected_model.named_parameters()))
        check('scalar_input_only_forward_inherited', tuple(inspect.signature(
            V9FastTCNPressureModel.forward).parameters) == ('self', 'x'))
        rejects('reject79_positions_unchanged', lambda: selected_model(x[:, :79]))
        rejects('reject_zero_length_unchanged', lambda: selected_model(x[:, :0]))
        rejects('reject_label_argument_unchanged', lambda: selected_model(x, target))
        rejects('reject_unsupported_convolution', lambda: _V9ShiftedLinearConv1d(nn.Conv1d(2, 3, 5, padding=2)))
        check('strict_determinism_never_weakened', torch.are_deterministic_algorithms_enabled()
            and not torch.is_deterministic_algorithms_warn_only_enabled()
            and not torch.backends.cuda.matmul.allow_tf32 and not torch.backends.cudnn.allow_tf32)
        return dict(schema='v9_fast_tcn_selfcheck', version=V9_FAST_TCN_VERSION,
            implementation_sha256=V9_FAST_TCN_CODE_SHA256, synthetic_only=True,
            python_version=platform.python_version(), torch_version=str(torch.__version__),
            selected_device=str(selected), oracle_device='cpu', oracle_dtypes=['float64', 'float32'],
            cuda_version=torch.version.cuda, parameter_count=int(parameter_count),
            tolerances=copy.deepcopy(V9_FAST_TCN_TOLERANCES), numerical_errors=errors,
            checks=checks, check_count=len(checks), passed=all(checks.values()),
            gpu_profile_or_engine_resume_proof=False)
    finally:
        _restore_rng(rng_before, generator)
        torch.set_num_threads(backend_before['threads'])
        torch.use_deterministic_algorithms(backend_before['deterministic'], warn_only=backend_before['warn_only'])
        torch.backends.cudnn.benchmark = backend_before['cudnn_benchmark']
        torch.backends.cudnn.deterministic = backend_before['cudnn_deterministic']
        torch.backends.cuda.matmul.allow_tf32 = backend_before['matmul_tf32']
        torch.backends.cudnn.allow_tf32 = backend_before['cudnn_tf32']
